# 04. Experiment Driver

[![Repo](https://img.shields.io/badge/GitHub-metaheuristic--budget--reproduction-181717?logo=github&logoColor=white)](https://github.com/prakash-ukhalkar/metaheuristic-budget-reproduction) [![License](https://img.shields.io/badge/license-MIT-green)](../LICENSE) [![Author](https://img.shields.io/badge/author-Prakash%20Ukhalkar-blue?logo=orcid&logoColor=white)](https://orcid.org/0000-0002-0452-6574) [![Python](https://img.shields.io/badge/python-3.10%2B-blue)](../requirements.txt)

**Source script:** `src/runner.py` &nbsp;|&nbsp; **Notebook 4 of 10**

Runs the `tune`, `main`, `tuned`, and `cons` experiment sweeps with paired seeding across algorithms.

Part of *Budget-Controlled Reproduction Study of Nature-Inspired Metaheuristics* — a reproduction study comparing six metaphor-based metaheuristics (GWO, WOA, SCA, SSA, HHO, AOA) against five established baselines (DE, PSO, L-SHADE, CMA-ES, random search) on constrained engineering design problems, under matched evaluation budgets and tuning effort.

See the [repository README](../README.md) for installation and full reproduction instructions, and [notebooks/README.md](README.md) for the notebook index and suggested run order.

---


# Experiment Driver

![Python](https://img.shields.io/badge/python-3.10%2B-blue) ![Status](https://img.shields.io/badge/status-research--reproduction-lightgrey) ![License](https://img.shields.io/badge/license-MIT-green)

Drives the four experiment phases: `tune` (matched-budget successive-halving tuner on a training subset), `main` (author-default hyperparameters across all algorithms and problems), `tuned` (tuned hyperparameters) and `cons` (constraint-handling sensitivity). Seeding is paired across algorithms so cross-algorithm comparisons are matched.


## Imports

External libraries and project modules used by this notebook.

In [ ]:
import json
import os
import sys
import time
import warnings

import numpy as np

In [ ]:
warnings.filterwarnings("ignore")

from problems import PROBLEMS, PROBLEM_MAP           # noqa: E402
from algorithms import ALGORITHMS, SPACES, DEFAULTS, run_one   # noqa: E402

BUDGET = 15000
N_RUNS = 51
TRAIN_PROBLEMS = ["PressureVessel", "CantileverBeam", "GearTrain"]
HELDOUT = [p.name for p in PROBLEMS if p.name not in TRAIN_PROBLEMS]
CONS_SUBSET = ["WeldedBeam", "SpeedReducer", "TensionSpring", "ThreeBarTruss"]
BUDGET_SUBSET = ["WeldedBeam", "SpeedReducer", "PressureVessel", "TensionSpring"]
TUNE_INIT = 32   # successive-halving racing budget
TUNE_SEEDS = 8

OUT = os.path.join(os.path.dirname(__file__), "..", "results")
os.makedirs(OUT, exist_ok=True)

### `sample_config`

Draws one random hyperparameter configuration for an algorithm from its search space.


In [ ]:
def sample_config(alg, rng):
    cfg = {}
    for k, spec in SPACES[alg].items():
        kind, lo, hi = spec
        cfg[k] = int(rng.integers(lo, hi + 1)) if kind == "int" else float(rng.uniform(lo, hi))
    return cfg

### `normalised_score`

Scale-free tuning score: relative gap to the published optimum, capped, with a penalty for infeasible runs.


In [ ]:
def normalised_score(res, problem):
    """Scale-free score: relative gap to the published best, +10 if infeasible."""
    if not res["feasible"]:
        return 10.0
    kf = problem.known_f
    denom = max(abs(kf), 1e-8)
    return min(10.0, abs(res["best_f"] - kf) / denom)

### `phase_tune`

Matched-budget successive-halving racing tuner: identical protocol cost for every algorithm.


In [ ]:
def phase_tune():
    """Matched-budget tuning by successive halving (a simple racing scheme).

    Every algorithm gets the identical protocol: 32 sampled configurations are
    raced on the three training problems, the worst half eliminated at each
    round while the per-configuration seed count doubles, until one survives.
    Total cost is identical across algorithms by construction.
    """
    rng = np.random.default_rng(20260801)
    best, log = {}, []
    for alg in ALGORITHMS:
        if not SPACES[alg]:
            best[alg] = {}
            continue
        cands = [sample_config(alg, rng) for _ in range(TUNE_INIT)]
        seeds = 1
        rnd = 0
        while len(cands) > 1:
            scores = []
            for ci, cfg in enumerate(cands):
                s = []
                for pn in TRAIN_PROBLEMS:
                    p = PROBLEM_MAP[pn]
                    for r in range(seeds):
                        res = run_one(alg, p, BUDGET, 777000 + 97 * rnd + 31 * ci + r,
                                      params=cfg)
                        s.append(normalised_score(res, p))
                scores.append(float(np.mean(s)))
            keep = np.argsort(scores)[:max(1, len(cands) // 2)]
            log.append(dict(alg=alg, round=rnd, n_configs=len(cands), seeds=seeds,
                            best_score=float(np.min(scores))))
            cands = [cands[i] for i in keep]
            seeds = min(TUNE_SEEDS, seeds * 2)
            rnd += 1
        best[alg] = cands[0]
        print(f"[tune] {alg:<8} cfg={cands[0]}", flush=True)
    json.dump(best, open(f"{OUT}/tuned_params.json", "w"), indent=2)
    json.dump(log, open(f"{OUT}/tuning_log.json", "w"), indent=2)

### `sweep`

Runs every algorithm on every problem for `N_RUNS` seeded repetitions and writes results to JSON, with optional resume.


In [ ]:
def sweep(tag, params_map, problems, scheme="deb"):
    fn = f"{OUT}/{tag}_{scheme}.json"
    rows = json.load(open(fn)) if os.path.exists(fn) and os.environ.get("RESUME") else []
    done = {r["problem"] for r in rows}
    t0 = time.time()
    for j, pn in enumerate(problems):
        if pn in done:
            continue
        p = PROBLEM_MAP[pn]
        for alg in ALGORITHMS:
            for r in range(N_RUNS):
                seed = 10000 * j + r
                res = run_one(alg, p, BUDGET, seed,
                              params=params_map.get(alg), scheme=scheme)
                rows.append(dict(phase=tag, scheme=scheme, problem=pn, algorithm=alg,
                                 run=r, seed=seed, feasible=res["feasible"],
                                 best_f=res["best_f"] if res["feasible"] else None,
                                 violation=res["best_viol"],
                                 trace=res["trace"]))
        json.dump(rows, open(fn, "w"))
        print(f"[{tag}/{scheme}] {pn} done  ({time.time()-t0:.0f}s)", flush=True)
    return rows

In [ ]:
if __name__ == "__main__":
    phase = sys.argv[1]
    if phase == "tune":
        phase_tune()
    elif phase == "main":
        sweep("main", {a: DEFAULTS.get(a, {}) for a in ALGORITHMS},
              [p.name for p in PROBLEMS])
    elif phase == "tuned":
        tp = json.load(open(f"{OUT}/tuned_params.json"))
        lim = int(sys.argv[2]) if len(sys.argv) > 2 else 99
        sweep("tuned", tp, [p.name for p in PROBLEMS][:lim])
    elif phase == "budget":
        d = {a: DEFAULTS.get(a, {}) for a in ALGORITHMS}
        import runner as _self
        for b in ([int(x) for x in (sys.argv[2].split(",") if len(sys.argv)>2 else ["5000","50000"])]):
            _self.BUDGET = b
            globals()["BUDGET"] = b
            rows = []
            for j, pn in enumerate(BUDGET_SUBSET):
                p = PROBLEM_MAP[pn]
                for alg in ALGORITHMS:
                    for r in range(15):
                        res = run_one(alg, p, b, 10000 * j + r, params=d.get(alg))
                        rows.append(dict(phase=f"budget{b}", scheme="deb", problem=pn,
                                         algorithm=alg, run=r, seed=10000 * j + r,
                                         feasible=res["feasible"],
                                         best_f=res["best_f"] if res["feasible"] else None,
                                         violation=res["best_viol"], trace=res["trace"]))
                print(f"[budget{b}] {pn} done", flush=True)
            json.dump(rows, open(f"{OUT}/budget{b}_deb.json", "w"))
    elif phase == "cons":
        d = {a: DEFAULTS.get(a, {}) for a in ALGORITHMS}
        for sc in ("static", "eps"):
            sweep("cons", d, CONS_SUBSET, scheme=sc)
    print("PHASE COMPLETE:", phase, flush=True)

---
## Key outcomes

- Seeds are paired across algorithms (`seed = 10000*j + r` for problem `j`, run `r`), which is what
  makes the paired Wilcoxon tests in notebook 06 valid.
- The `tune` phase uses successive-halving racing (32 configurations, doubling seed count per round) so
  every algorithm receives an identical tuning budget; SSA and AOA were found to overfit the 3-problem
  training set (manuscript caveat).
- `main` and `tuned` sweeps each produce 51 runs x 11 algorithms x 9 problems = 5,049 result rows; the
  `cons` phase repeats a 4-problem subset under the static-penalty and epsilon-constrained schemes for
  sensitivity analysis (notebook 06/08).

*Part of the budget-controlled reproduction study of nature-inspired metaheuristics.*


---

**Author:** Prakash Ukhalkar ([ORCID: 0000-0002-0452-6574](https://orcid.org/0000-0002-0452-6574)) — Pimpri Chinchwad College of Engineering, Pune, India

**Repository:** [github.com/prakash-ukhalkar/metaheuristic-budget-reproduction](https://github.com/prakash-ukhalkar/metaheuristic-budget-reproduction) &nbsp;|&nbsp; **License:** [MIT](../LICENSE) &nbsp;|&nbsp; **Citation:** [CITATION.cff](../CITATION.cff)

[![Repo](https://img.shields.io/badge/GitHub-metaheuristic--budget--reproduction-181717?logo=github&logoColor=white)](https://github.com/prakash-ukhalkar/metaheuristic-budget-reproduction) [![License](https://img.shields.io/badge/license-MIT-green)](../LICENSE)
